# Financial Document Intelligence System

**Author:** Siddharth Khedekar

An end-to-end retrieval pipeline for financial documents: structure-aware
segmentation, financial NER, keyphrase extraction, FinBERT embeddings, and
FAISS-based semantic retrieval — following the architecture pattern used in
production financial-document search tools.

## Pipeline overview

```
Raw document -> Segmentation -> NER + Keyphrase extraction -> FinBERT embedding
             -> FAISS index -> Query-time semantic retrieval
```

## Data note
This notebook runs on small **synthetic sample documents** (not real filings)
bundled in `data/sample_filings/`, so it runs end-to-end without requiring any
download. `src/data_ingestion.py` provides a utility to pull real filings from
SEC EDGAR (free, public) to scale this to a real corpus — see the final
section for how that fits in.


## 1. Setup

Importing the modular pipeline components from `src/` rather than writing
everything as notebook cells — this mirrors how a real production system
would be structured (reusable, testable modules), and keeps this notebook
focused on demonstrating and explaining the pipeline rather than
implementation detail.


In [1]:
import sys
sys.path.append("..")

from src.segmentation import segment_by_section_headers, segment_by_speaker
from src.ner_extraction import FinancialNER, extract_financial_figures
from src.keyword_extraction import ThemeExtractor
from src.embedding_index import FinBertEmbedder, RetrievalIndex, IndexedChunk

import numpy as np

ModuleNotFoundError: No module named 'keybert'

## 2. Load Sample Documents

Two document types are included, since real-world financial NLP systems
need to handle both filing structure and transcript structure:
- A synthetic 10-K style filing (section-structured)
- A synthetic earnings call transcript (speaker-turn structured)


In [ ]:
with open("../data/sample_filings/sample_10k_synthetic.txt") as f:
    filing_text = f.read()

with open("../data/sample_filings/sample_earnings_call_synthetic.txt") as f:
    transcript_text = f.read()

print(f"Filing length: {len(filing_text)} characters")
print(f"Transcript length: {len(transcript_text)} characters")

## 3. Segmentation

**Why different segmentation strategies per document type:** a filing has
named sections (Risk Factors, MD&A) that are meaningful retrieval units on
their own. A transcript's meaningful units are speaker turns — merging an
analyst's question with the executive's answer into one chunk would blur
exactly the distinction an analyst usually cares about (what was asked vs.
what was actually said).


In [ ]:
SECTION_HEADERS = [
    "Item 1A. Risk Factors",
    "Item 7. Management's Discussion and Analysis of Financial Condition and Results of Operations",
    "Item 9A. Controls and Procedures",
]

filing_chunks = segment_by_section_headers(filing_text, source_doc="sample_10k", headers=SECTION_HEADERS)
transcript_chunks = segment_by_speaker(transcript_text, source_doc="sample_earnings_call")

print(f"Filing segmented into {len(filing_chunks)} section-based chunks")
print(f"Transcript segmented into {len(transcript_chunks)} speaker-turn chunks\n")

print("Example filing chunk:")
print(f"  Section: {filing_chunks[0].section}")
print(f"  Text: {filing_chunks[0].text[:150]}...\n")

print("Example transcript chunk:")
print(f"  Speaker: {transcript_chunks[0].section}")
print(f"  Text: {transcript_chunks[0].text[:150]}...")

## 4. Named Entity Recognition + Financial Figure Extraction

**Why a hybrid approach:** the Transformer NER model handles organizations,
people, and locations well, but general-purpose NER models are not reliably
trained to correctly tag monetary values and percentages as a distinct type.
Regex extraction is deliberately layered on top for exactly those two
categories — using the right tool per entity type rather than forcing one
method to do everything.


In [ ]:
ner = FinancialNER()

all_chunks = filing_chunks + transcript_chunks

print("Extracting entities from all chunks (this loads a Transformer model on first run)...\n")
for chunk in all_chunks[:3]:  # preview first few for readability
    entities = ner.extract(chunk.text)
    figures = extract_financial_figures(chunk.text)
    print(f"[{chunk.chunk_id}] ({chunk.section})")
    print(f"  Entities: {[(e.text, e.label) for e in entities]}")
    print(f"  Figures:  {[(f.text, f.label) for f in figures]}\n")

## 5. Theme / Keyphrase Extraction (KeyBERT)

**Why embedding-similarity-based extraction over frequency-based (TF-IDF):**
the most frequent words in a filing are usually boilerplate ("Company,"
"fiscal year"). KeyBERT scores candidate phrases by similarity to the whole
chunk's meaning, which surfaces the actual substantive themes
("supply chain disruption," "margin compression") instead.


In [ ]:
theme_extractor = ThemeExtractor()

for chunk in filing_chunks:
    themes = theme_extractor.extract_themes(chunk.text, top_n=5)
    print(f"[{chunk.chunk_id}] ({chunk.section})")
    print(f"  Themes: {themes}\n")

## 6. FinBERT Embeddings + FAISS Index

**Why FinBERT specifically, not a generic sentence encoder:** FinBERT is
pretrained on financial-domain text, so its embedding space places
financially related concepts (e.g. "margin compression" and "gross margin
decline") closer together than a generic encoder would. This directly
improves retrieval precision for financial queries — a deliberate,
justifiable architecture choice rather than a default.

**Why `IndexFlatL2` at this scale:** exact nearest-neighbor search, no
approximation — the right choice for a small prototype corpus. At real
production scale (100k+ chunks) this would move to an IVF/HNSW index to
keep search sub-linear; noted here as a scaling decision rather than
implemented prematurely for a corpus this small.


In [ ]:
embedder = FinBertEmbedder()

texts = [c.text for c in all_chunks]
print("Embedding all chunks with FinBERT (first run downloads the model)...")
embeddings = embedder.embed(texts)
print(f"Embedding matrix shape: {embeddings.shape}")

index = RetrievalIndex(embedding_dim=embeddings.shape[1])
indexed_chunks = [
    IndexedChunk(chunk_id=c.chunk_id, text=c.text, source_doc=c.source_doc, section=c.section)
    for c in all_chunks
]
index.add(embeddings, indexed_chunks)
print(f"FAISS index built with {len(indexed_chunks)} chunks")

## 7. Query-Time Retrieval

This is the actual user-facing capability: a natural-language query is
embedded with the same FinBERT model, then matched against the indexed
chunks — retrieving the most semantically relevant sections/speaker-turns
across both the filing and the transcript, regardless of which document
they came from.


In [ ]:
def query_documents(query: str, top_k: int = 3):
    query_embedding = embedder.embed([query])
    results = index.search(query_embedding, top_k=top_k)
    print(f'Query: "{query}"\n')
    for chunk, distance in results:
        print(f"  [{chunk.source_doc} | {chunk.section}] (distance={distance:.3f})")
        print(f"  {chunk.text[:200]}...\n")
    return results

_ = query_documents("What is driving margin pressure?")

In [ ]:
_ = query_documents("What did the CFO say about cybersecurity investment?")

## 8. Next Steps — Scaling Beyond the Prototype

1. **Real data**: use `src/data_ingestion.py` to pull actual 10-K/10-Q
   filings from SEC EDGAR (free, public, no gated access) instead of the
   bundled synthetic samples — the segmentation/NER/embedding pipeline
   above requires no changes to work on real filings.
2. **ANN index at scale**: swap `IndexFlatL2` for an IVF or HNSW FAISS
   index once the corpus grows into the tens of thousands of chunks, to
   keep retrieval latency sub-linear.
3. **Domain-tuned NER**: fine-tune the NER model on a financial-NER-labeled
   dataset (e.g. FiNER) to distinguish financial-specific entity types
   (TICKER, monetary categories) beyond generic PERSON/ORG/DATE.
4. **LLM synthesis layer**: feed retrieved chunks into an LLM (e.g. Groq
   LLaMA, consistent with prior project work) to generate a grounded,
   cited natural-language answer instead of returning raw chunks.
5. **Evaluation**: build a small labeled query-relevance set to measure
   retrieval precision/recall — critical for justifying design choices
   (e.g. FinBERT vs. generic encoder) with actual numbers rather than
   only architectural reasoning.
